In [1]:
import sys
sys.path.append("/home/ubuntu/ins-plt/DS-Inference-Scripts/rl_scripts/retrain/")

In [2]:
from core import *
from typing import List, Dict
from eq_common_utils.utils.config.s3_config import s3_config

s3_helper = s3_config()
s3 = s3_helper._s3Client

/tmp
######################## S3Connection ######################
######################## S3Connection ######################
PreprodConfig(s3_bucket='eq-model-output', s3_model_bucket_key='eq-training-dump', s3_destination_data_bucket='eq-model-output', s3_source_data_bucket='eq-model-output', s3_predictions_prefix='ttm_model/schedular/date/predictions/', s3_metrics_prefix='ttm_model/schedular/date/metrics/', s3_model_prefix='ttm_model/schedular/data_dump/model_year/', s3_features_prefix='ttm_model/schedular/date/features/', s3_source_data_prefix='ttm_model/data_dump/', s3_destination_data_prefix='ttm_model/data_dump/', s3_meta_data_prefix='ttm_model/schedular/date/', data_sources=DataSourceConfig(closeprice='eq_financial_model', expected_return='eq_er_model', model_predictions='eq_ttm_model'), model_settings=ModelConfig(model_year='2024_Q4', ttm_model_revision='1024_96_v1', context_length=1024, buffer_length=2000, head_dropout_rate=0.1), metrics_config=MetricsConfig(req_columns=['is

In [19]:
exchange_map = {
    "UN": "",
    "UW": "",
    "UQ": "",
    "UR": "",
    "UU": "",
    "UV": "",
    "NO": ".OL",
    "SS": ".SS",
    "LX": ".LU",
    "BB": ".BR",
    "GY": ".DE",
    "HK": ".HK",
    "IM": ".MI",
    "SE": ".ST",
    "IT": ".MI",
    "JT": ".T",
    "FH": ".HE",
    "LN": ".L",
    "NZ": ".NZ",
    "SP": ".SI",
    "SJ": ".JO",
    "DC": ".CO",
    "PW": ".WA",
    "FP": ".PA",
    "NA": ".AS",
    "SQ": ".SI",
    "ID": ".JK",
    "AV": ".VI",
    "PL": ".PS",
    "AT": ".AT",
    "IJ": ".JK",
    "LI": ".LS",
    "UF": "",
    "VX": ".SW",
    "CV": ".V",
    "JQ": ".T",
    "NS": ".NS",
    "CT": ".TO",
    "SF": ".SW",
    "PM": ".OTC",
    "Pfd": ""
}

def bloomberg_to_yahoo(bb_ticker):

    parts = bb_ticker.split()

    if len(parts) < 2:
        return None

    symbol = parts[0]
    exch = parts[1]

    suffix = exchange_map.get(exch, "")

    return symbol + suffix


In [20]:
benchmark_path = "/home/ubuntu/ins-plt/DS-Inference-Scripts/rl_scripts/retrain/HSBC/GLOBAL AIPEX/gbs_solactive.csv"
benchmark = pd.read_csv(benchmark_path)
benchmark['Ticker'] = benchmark['BBEx'].apply(bloomberg_to_yahoo)

In [21]:
sector_missing = pd.read_csv("sector_missing.csv")
benchmark_missing = benchmark[benchmark['ISIN'].isin(sector_missing['isin'].tolist())]
missing_tickers = benchmark_missing['Ticker'].tolist()

In [22]:
import yfinance as yf
import pandas as pd
import time

def get_sector_info(tickers, delay=0.5):
    records = []

    for tk in tqdm(tickers):
        try:
            info = yf.Ticker(tk).get_info()

            records.append({
                "ticker": tk,
                "sector": info.get("sector"),
                "industry": info.get("industry")
            })
            print(records[-1])

        except Exception as e:
            records.append({
                "ticker": tk,
                "sector": None,
                "industry": None,
                "error": str(e)
            })
            break

        time.sleep(delay)   # <-- throttling

    return pd.DataFrame(records)


In [23]:
benchmark['BBEx'].apply(lambda bbex: bbex.split(' ')[1].strip()).unique()

array(['NO', 'SS', 'LX', 'BB', 'UN', 'GY', 'HK', 'IM', 'SE', 'IT', 'UW',
       'CT', 'JT', 'FH', 'LN', 'NZ', 'UV', 'UQ', 'SP', 'SJ', 'DC', 'PW',
       'FP', 'NA', 'UR', 'SQ', 'ID', 'AV', 'PL', 'AT', 'IJ', 'LI', 'UF',
       'VX', 'CV', 'JQ', 'NS', 'UU', 'SF', 'Pfd', 'PM'], dtype=object)

In [24]:
benchmark_missing[benchmark_missing['ISIN'] == 'US5312298137']

,ISIN,SEDOL,BBEx,CompanyName,2006-05-31,2006-06-30,2006-07-31,2006-08-31,2006-09-29,2006-10-31,...,2025-04-30,2025-05-30,2025-06-30,2025-07-31,2025-08-29,2025-09-30,2025-10-31,2025-11-28,2025-12-31,Ticker
92,US5312298137,BPLYVJ1,LSXMA UW Equity,LIBERTY MEDIA CORP - SIRIUSXM A,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,LSXMA


In [25]:
df = get_sector_info(missing_tickers)
print(df)

  0%|          | 0/492 [00:00<?, ?it/s]HTTP Error 404: 


{'ticker': 'SECTB.SS', 'sector': None, 'industry': None}


  0%|          | 1/492 [00:00<05:29,  1.49it/s]HTTP Error 404: 


{'ticker': '283.HK', 'sector': None, 'industry': None}


  0%|          | 2/492 [00:01<05:29,  1.49it/s]

...

{'ticker': 'LSXMA', 'sector': None, 'industry': None}


  2%|▏         | 8/492 [00:04<04:53,  1.65it/s]HTTP Error 404: 


{'ticker': 'BPY-U.TO', 'sector': None, 'industry': None}


  2%|▏         | 8/492 [00:05<05:29,  1.47it/s]


KeyboardInterrupt: 

In [28]:
benchmark_missing

,ISIN,SEDOL,BBEx,CompanyName,2006-05-31,2006-06-30,2006-07-31,2006-08-31,2006-09-29,2006-10-31,...,2025-04-30,2025-05-30,2025-06-30,2025-07-31,2025-08-29,2025-09-30,2025-10-31,2025-11-28,2025-12-31,Ticker
1,SE0022419784,BRC3MR6,SECTB SS Equity,SECTRA AB-B SHS,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000062,0.000057,0.000053,0.000051,0.000045,SECTB.SS
12,HK0283012463,6680440,283 HK Equity,GOLDIN PROPERTIES HOLDINGS LTD,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,283.HK
38,US09075E1001,BDTYZ53,BIVV UW Equity,BIOVERATIV INC,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,BIVV
52,US7475451016,BYNWVQ3,QCP UN Equity,QUALITY CARE PROPERTIES,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,QCP
71,US20084V3069,BZ76T20,CHUBK UQ Equity,COMMERCEHUB INC - SERIES C,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,CHUBK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3261,US83443Q1031,BW2K5V7,SOLS UW Equity,SOLSTICE ADVANCED MATERIALS INC,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000092,0.000096,0.000097,SOLS
3262,SGXE35825913,BVSYC71,8YZ SP Equity,YANGZIJIANG MARITIME DEVELOPMENT LTD,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,8YZ.SI
3263,DE000A40ZVV0,BT7MHJ9,CECV GY Equity,CECONOMY AG (TENDERED SHARES),0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,CECV.DE
3264,IL0012327503,BVMBP91,ORNP IT Equity,ORION COMMERCIAL ASSETS LTD,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,ORNP.MI


In [47]:
import yfinance as yf
import re

import re

def reduce_company_name(name):

    if not isinstance(name, str):
        return name

    suffix_patterns = [
        r'\bHOLDINGS?\b',
        r'\bLIMITED\b',
        r'\bLTD\b',
        r'\bCORPORATION\b',
        r'\bCORP\b',
        r'\bGROUP\b',
        r'\bPLC\b',
        r'\bINC\b',
        r'\bSA\b',
        r'\bNV\b',
        r'\bAG\b'
    ]

    for pattern in suffix_patterns:
        name = re.sub(pattern, '', name, flags=re.IGNORECASE)

    name = re.sub(r'\s+', ' ', name)

    return name.strip()


import re

import re

def normalize_company_name(name):

    if not isinstance(name, str):
        return name

    name = reduce_company_name(name)

    # Remove Class patterns
    name = re.sub(r'[-\s]*Class\s+[A-Z0-9]+', '', name, flags=re.IGNORECASE)

    # Remove Series patterns
    name = re.sub(r'[-\s]*Series\s+[A-Z0-9]+', '', name, flags=re.IGNORECASE)

    # Remove vendor share suffix patterns
    name = re.sub(r'[-\s]*[A-Z]\s+SHS?', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\bSHS?\b', '', name, flags=re.IGNORECASE)

    # Remove share type descriptors
    name = re.sub(r'\bOrdinary\b.*', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\bCommon\b.*', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\bPreferred\b.*', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\bPreference\b.*', '', name, flags=re.IGNORECASE)

    # 🔥 NEW: Remove trailing separators
    name = re.sub(r'[-/,:\s]+$', '', name)

    # Cleanup whitespace
    name = re.sub(r'\s+', ' ', name)
    print(name)

    return name.strip()



def search_by_company(company_name):
    try:
        results = yf.Search(normalize_company_name(company_name))
        return results.quotes
    except:
        return []

In [78]:
mapping = {
    'Healthcare': 'Health Care',
    'Real Estate': 'Real Estate',
    'Technology': 'Information Technology',
    'Communication Services': 'Communication Services',
    'Financials': 'Financials',
    'Industrials': 'Industrials',
    'Basic Materials': 'Materials',
    'Consumer Discretionary': 'Consumer Discretionary',
    'Energy': 'Energy',
    'Utilities': 'Utilities',
    'Consumer Staples': 'Consumer Staples',
    'Consumer Defensive': 'Consumer Staples',
    'Financial Services': 'Financials',
    'Energy / Industrials': 'Energy',
    'Consumer Cyclical': 'Consumer Discretionary',
    'Technology / Energy': 'Information Technology',
    'Non-Profit / Finance': 'Financials',
    'Real Estate / Materials': 'Real Estate',
    'Industrials / Tech': 'Industrials'
}


missing_sector_map = pd.read_csv("missing_sector_map.csv", encoding="latin1")
missing_sector_map['Sector'] = missing_sector_map['Sector'].map(mapping)
missing_sector_map = dict(zip(missing_sector_map['isin'].tolist(), missing_sector_map['Sector'].tolist()))

In [79]:
def transpose(data, index_col, new_index_col):
    data = data.set_index(index_col).T.reset_index().rename(columns={'index': new_index_col})
    data.index.name = None
    data.columns.name = None
    return data

In [80]:
benchmark_path = "/home/ubuntu/ins-plt/DS-Inference-Scripts/rl_scripts/retrain/HSBC/GLOBAL AIPEX/gbs_solactive.csv"
benchmark = pd.read_csv(benchmark_path).drop(columns=['CompanyName', 'SEDOL', 'BBEx'])
benchmark = benchmark.groupby('ISIN').sum().reset_index()

rebalance_dates = benchmark.columns[1:]
benchmark = benchmark.rename(columns={date: pd.Timestamp(date).strftime('%Y-%m-%d') for date in rebalance_dates})
benchmark = transpose(benchmark, 'ISIN', 'date')

In [82]:
sector_isins = pd.concat([get_master('qst1'), get_master('qst2'), get_master('qst3'), get_master('all')]).drop_duplicates(subset=['isin']).reset_index(drop=True)[['isin', 'ind_code']]

sector_map = {
    "10": "Energy",
    "15": "Materials",
    "20": "Industrials",
    "25": "Consumer Discretionary",
    "30": "Consumer Staples",
    "35": "Health Care",
    "40": "Financials",
    "45": "Information Technology",
    "50": "Communication Services",
    "55": "Utilities",
    "60": "Real Estate",
}

sector_isins = sector_isins[~sector_isins['ind_code'].isna()].reset_index(drop=True)
sector_isins['Sector'] = sector_isins['ind_code'].apply(lambda ind_code: sector_map.get(str(ind_code)[:2], 'NA'))
sector_map = dict(zip(sector_isins['isin'].tolist(), sector_isins['Sector'].tolist()))

In [83]:
sector_map.update(missing_sector_map)

In [89]:
sector_filtered = benchmark.set_index('date').loc[rebalance_dates].fillna(0.0).reset_index()
sector_filtered = transpose(sector_filtered, 'index', 'isin')
sector_filtered['sector_name'] = sector_filtered['isin'].map(sector_map)
bench_dates = sector_filtered.drop(columns=['isin']).columns[:-1].tolist()

sector_filtered = sector_filtered[['sector_name'] + bench_dates].groupby(by='sector_name').sum().reset_index()
sector_filtered = sector_filtered[sector_filtered['sector_name'] != 'NA']

sector_temp = sector_filtered.set_index('sector_name').T
sector_filtered = sector_temp.div(sector_temp.sum(axis=1), axis='index').T
sector_filtered = sector_temp.T

In [95]:
def get_isin_sweight(date, sector):
    return sector_filtered.loc[sector][date]

isins = benchmark.columns.tolist()[1:]
bench_data = pd.DataFrame({'date': bench_dates})
for isin in isins:
    sector = sector_map.get(isin, 'NA')
    bench_data[isin] = bench_data['date'].apply(lambda date: get_isin_sweight(date, sector))

/tmp/ipykernel_785326/1398817063.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  bench_data[isin] = bench_data['date'].apply(lambda date: get_isin_sweight(date, sector))
/tmp/ipykernel_785326/1398817063.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  bench_data[isin] = bench_data['date'].apply(lambda date: get_isin_sweight(date, sector))
/tmp/ipykernel_785326/1398817063.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  

In [99]:
sector_weights_filtered = transpose(bench_data, 'date','isin')
sector_weights_filtered['sector_name'] = sector_weights_filtered['isin'].map(sector_map).fillna("NA")
sector_weights_filtered.to_csv("data_processed/sector_weights.csv", index=False)

In [ ]:
se

array(['Energy', 'Communication Services', 'Industrials', 'Financials',
       'Health Care', 'Real Estate', 'Materials', 'Utilities',
       'Consumer Discretionary', 'Information Technology',
       'Consumer Staples'], dtype=object)